# Image Generation Project — A1111 on Google Colab

Run the single code cell below in a **GPU** Colab runtime. On its first run it mounts Drive, creates the project, installs WebUI and downloads only missing assets. Later runs reuse the WebUI, checkpoint, LoRAs, ControlNet models, configuration, and outputs saved in Drive.

> A runtime reconnect always clears GPU memory, so the checkpoint must load from Drive into GPU memory again. It is **not downloaded again**. A fresh public Gradio URL is also required for every new runtime.

## First-run credential

The first run may need a CivitAI token for its downloads. In Colab, open the key icon in the left sidebar, add a secret named `CIVITAI_TOKEN`, and enable notebook access. The token is read only when a required Drive file is missing; it is never written to Drive or printed.

Use only reference images you created or are authorised to use. This notebook does not add a prompt filter; service terms, applicable law, and model licences still apply.

In [ ]:
import os
from getpass import getpass

os.environ["CIVITAI_TOKEN"] = ""
os.environ["HF_TOKEN"] = ""
print("CivitAI token loaded for this session.")

CivitAI token loaded for this session.


✅ CLIP installed successfully.


In [ ]:
# ONE-CELL INSTALL / REUSE / LAUNCH
# First run: installs only missing components. Later runs: reuses Drive files and launches directly.
import json, os, re, shutil, subprocess, sys, time
# Colab injects an inline Matplotlib backend that is incompatible with A1111's pinned Matplotlib.
# Set a non-interactive backend before any package validation/import occurs.
os.environ['MPLBACKEND'] = 'Agg'
# This notebook uses CUDA Torch, xFormers and half-precision SD 1.5 weights. Fail immediately
# on a CPU/TPU runtime instead of spending time restoring or installing packages first.
gpu_check = subprocess.run(['nvidia-smi'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL) if shutil.which('nvidia-smi') else None
if gpu_check is None or gpu_check.returncode != 0:
    raise RuntimeError('NVIDIA GPU NOT AVAILABLE. In Colab choose Runtime > Change runtime type > Hardware accelerator: T4 GPU, save, reconnect, and rerun this cell.')
from pathlib import Path
from getpass import getpass

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

PROJECT_DIR = Path('/content/drive/MyDrive/Image_Generation_Project')
WEBUI_DIR = PROJECT_DIR / 'webui'
MODELS_DIR = PROJECT_DIR / 'models'
LORAS_DIR = PROJECT_DIR / 'loras'
OUTPUTS_DIR = PROJECT_DIR / 'outputs'
CONTROLNET_DIR = PROJECT_DIR / 'controlnet_models'
PERSON_MODELS_DIR = PROJECT_DIR / 'person_models'
REFERENCE_DIR = PROJECT_DIR / 'reference_images'
# Keep installed packages on Colab's fast temporary disk, never in Drive.
RUNTIME_VENV_DIR = Path('/content/image_generation_project_venv')
VENV_ARCHIVE = PROJECT_DIR / 'runtime_cache' / 'a1111_venv_py310.tar'
for folder in [PROJECT_DIR, MODELS_DIR, LORAS_DIR, OUTPUTS_DIR, CONTROLNET_DIR, PERSON_MODELS_DIR, REFERENCE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
for name in ('identity', 'style', 'clothing', 'pose'):
    (REFERENCE_DIR / name).mkdir(exist_ok=True)

def run(command, *, cwd=None):
    print('\n$', ' '.join(map(str, command)))
    subprocess.run(list(map(str, command)), cwd=str(cwd) if cwd else None, check=True)

def nonempty(path, minimum_bytes=1024):
    return path.exists() and path.is_file() and path.stat().st_size >= minimum_bytes

# Git is used only on the FIRST run, when the Drive-hosted WebUI folder does not exist.
WEBUI_TAG = 'v1.10.1'
CONTROLNET_REPO = 'https://github.com/Mikubill/sd-webui-controlnet.git'
CONTROLNET_COMMIT = '56cec5b2958edf3b1807b7e7b2b1b5186dbd2f81'
# A1111 v1.10.1 references an upstream repository that was deleted. This public mirror
# retains the compatible source; its pinned code exposes BasicTransformerBlock.ATTENTION_MODES.
SD_REPO = 'https://github.com/joypaul162/Stability-AI-stablediffusion.git'
SD_COMMIT = 'f16630a927e00098b524d687640719e4eb469b76'

if not (WEBUI_DIR / '.git').exists():
    if WEBUI_DIR.exists() and any(WEBUI_DIR.iterdir()):
        raise RuntimeError(f'{WEBUI_DIR} exists but is not a WebUI Git checkout. Rename it in Drive, then rerun.')
    print('First run: cloning A1111 into Drive...')
    run(['git', 'clone', '--branch', WEBUI_TAG, '--depth', '1', 'https://github.com/AUTOMATIC1111/stable-diffusion-webui.git', WEBUI_DIR])
else:
    print('Reusing saved A1111 WebUI from Drive — no Git update or checkout will run.')

controlnet_extension = WEBUI_DIR / 'extensions' / 'sd-webui-controlnet'
if not (controlnet_extension / '.git').exists():
    print('First run: installing ControlNet extension...')
    run(['git', 'clone', CONTROLNET_REPO, controlnet_extension])
    run(['git', '-C', controlnet_extension, 'checkout', '--detach', CONTROLNET_COMMIT])
else:
    print('Reusing saved ControlNet extension from Drive.')

# Install all A1111 source repositories explicitly. Normal reruns are read-only; the one
# incompatible historical Stable Diffusion dependency is replaced after validation.
def ensure_repository(destination, repository_url, commit, repair_existing=False, required_file=None, required_text=None):
    def safe_remove_dependency_tree(path):
        expected_parent = destination.parent.resolve()
        resolved = path.resolve()
        allowed = (path.name == destination.name or path.name.startswith(destination.name + '_compatible_staging_') or path.name.startswith(destination.name + '_incompatible_backup_'))
        if resolved.parent != expected_parent or not allowed:
            raise RuntimeError(f'Refusing to delete unexpected path: {path}')
        if path.is_symlink(): path.unlink()
        elif path.exists(): shutil.rmtree(path)
        print(f'Removed obsolete dependency files: {path}')
    def source_is_valid():
        if not required_file: return True
        check_file = destination / required_file
        return check_file.exists() and (not required_text or required_text in check_file.read_text(encoding='utf-8', errors='ignore'))
    if repair_existing:
        for pattern in (destination.name + '_compatible_staging_*', destination.name + '_incompatible_backup_*'):
            for stale in destination.parent.glob(pattern): safe_remove_dependency_tree(stale)
    if (destination / '.git').exists():
        current = subprocess.run(['git', '-C', str(destination), 'rev-parse', 'HEAD'], text=True, capture_output=True).stdout.strip()
        if not repair_existing or (current == commit and source_is_valid()):
            print(f'Reusing saved dependency: {destination.name}')
            return
        # Do not fetch, reset, or check out inside an incompatible saved repository: its Git
        # metadata may contain locks or corruption. Build a validated replacement separately.
        print(f'Building a clean compatible replacement for: {destination.name}')
        stamp = time.strftime('%Y%m%d_%H%M%S')
        staging = destination.parent / f'{destination.name}_compatible_staging_{stamp}'
        if staging.exists():
            raise RuntimeError('Generated dependency repair paths already exist; wait one second and rerun.')
        run(['git', 'clone', '--depth', '1', repository_url, staging])
        run(['git', '-C', staging, 'checkout', '--detach', commit])
        staged_file = staging / required_file if required_file else None
        staged_valid = not staged_file or (staged_file.exists() and (not required_text or required_text in staged_file.read_text(encoding='utf-8', errors='ignore')))
        if not staged_valid:
            raise RuntimeError(f'Fresh compatible source failed validation; original dependency remains untouched at {destination}.')
        safe_remove_dependency_tree(destination)
        staging.rename(destination)
        print(f'Compatible dependency installed; obsolete source was deleted: {destination}')
        return
    if destination.exists() and any(destination.iterdir()):
        if repair_existing: safe_remove_dependency_tree(destination)
        else: raise RuntimeError(f'Dependency folder exists but is incomplete: {destination}. Do not delete models; repair only this dependency folder.')
    destination.parent.mkdir(parents=True, exist_ok=True)
    print(f'First run: cloning dependency {destination.name}...')
    run(['git', 'clone', repository_url, destination])
    checkout = subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', commit], text=True)
    if checkout.returncode != 0 and repair_existing:
        raise RuntimeError(f'Could not check out required revision {commit} for {destination.name}.')
    if checkout.returncode != 0:
        print(f'Pinned revision unavailable for {destination.name}; using its freshly cloned HEAD. SD 1.5 generation does not use the optional SDXL repository.')
    if not source_is_valid():
        raise RuntimeError(f'{destination.name} failed its A1111 compatibility check after cloning.')

repositories_dir = WEBUI_DIR / 'repositories'
ensure_repository(repositories_dir / 'stable-diffusion-webui-assets', 'https://github.com/AUTOMATIC1111/stable-diffusion-webui-assets.git', '6f7db241d2f8ba7457bac5ca9753331f0c266917')
ensure_repository(repositories_dir / 'stable-diffusion-stability-ai', SD_REPO, SD_COMMIT, repair_existing=True, required_file='ldm/modules/attention.py', required_text='ATTENTION_MODES')
ensure_repository(repositories_dir / 'generative-models', 'https://github.com/Stability-AI/generative-models.git', '45c443b316737a4ab6e40413d7794a7f5657c19f')
ensure_repository(repositories_dir / 'k-diffusion', 'https://github.com/crowsonkb/k-diffusion.git', 'ab527a9a6d347f364e3d185ba6d714e22d80cb3c')
ensure_repository(repositories_dir / 'BLIP', 'https://github.com/salesforce/BLIP.git', '48211a1594f1321b00f14c9f7a5b4813144b2fb9')
# The PyPI/pip build named taming-transformers can install metadata without the actual
# taming module. Keep the verified source in Drive and expose it directly to Python.
TAMING_DIR = repositories_dir / 'taming-transformers'
ensure_repository(TAMING_DIR, 'https://github.com/CompVis/taming-transformers.git', '24268930bf1dce879235a7fddd0b2355b84d7ea6')
os.environ['PYTHONPATH'] = str(TAMING_DIR) + os.pathsep + os.environ.get('PYTHONPATH', '')

# A1111 needs Python 3.10/3.11. Active packages live on fast temporary /content storage.
venv_python = RUNTIME_VENV_DIR / 'bin' / 'python'
environment_changed = False
if not venv_python.exists() and VENV_ARCHIVE.exists():
    print('Restoring the cached WebUI package environment from Drive to /content...')
    if RUNTIME_VENV_DIR.exists(): shutil.rmtree(RUNTIME_VENV_DIR)
    run(['tar', '-C', '/content', '-xf', VENV_ARCHIVE])
if not venv_python.exists():
    print('Creating the temporary local Python 3.10 WebUI environment in /content...')
    if RUNTIME_VENV_DIR.exists(): shutil.rmtree(RUNTIME_VENV_DIR)
    run(['bash', '-lc', 'apt-get update -qq && apt-get install -y -qq python3.10 python3.10-venv'])
    run(['/usr/bin/python3.10', '-m', 'venv', RUNTIME_VENV_DIR])
    run([venv_python, '-m', 'pip', 'install', 'pip==24.0', 'setuptools==69.5.1', 'wheel'])
    environment_changed = True
else:
    print('Reusing local /content WebUI packages for this runtime.')

def probe(code):
    return subprocess.run([venv_python, '-c', code], text=True, capture_output=True, env=os.environ.copy())

def installed_version(distribution):
    result = probe(f'import importlib.metadata as m; print(m.version({distribution!r}))')
    return result.stdout.strip() if result.returncode == 0 else None

def version_matches(actual, expected):
    return actual == expected or (actual is not None and actual.startswith(expected + '+'))

def install_if_needed(distribution, expected, spec=None, import_code=None, extra_args=None):
    global environment_changed
    actual = installed_version(distribution)
    import_result = probe(import_code) if import_code else None
    version_ok = version_matches(actual, expected)
    import_ok = import_result is None or import_result.returncode == 0
    if version_ok and import_ok:
        print(f'Reusing package: {distribution} {actual}')
        return
    package_spec = spec or f'{distribution}=={expected}'
    command = [venv_python, '-m', 'pip', 'install']
    if version_ok and not import_ok:
        print(f'Repairing broken binary/import for {distribution} {actual}...')
        command += ['--force-reinstall', '--no-deps']
    else:
        print(f'Installing required {distribution} {expected}; found {actual or "missing"}.')
        command += ['--upgrade']
    command += [package_spec] + list(extra_args or [])
    run(command)
    environment_changed = True

# Install Torch first. The official requirements file contains a bare `torch` line; we
# deliberately exclude it below so pip cannot silently replace this CUDA-compatible build.
install_if_needed('torch', '2.1.2', 'torch==2.1.2', 'import torch; assert torch.__version__.startswith("2.1.2")', ['--index-url', 'https://download.pytorch.org/whl/cu121'])
install_if_needed('torchvision', '0.16.2', 'torchvision==0.16.2', 'import torchvision; assert torchvision.__version__.startswith("0.16.2")', ['--index-url', 'https://download.pytorch.org/whl/cu121'])

# Check every pinned A1111 requirement and install only missing/mismatched distributions.
# Core binary packages are handled separately so their imports can also be tested.
excluded = {'torch', 'torchvision', 'numpy', 'scikit-image', 'open-clip-torch', 'setuptools'}
missing_specs = []
for raw_line in (WEBUI_DIR / 'requirements_versions.txt').read_text(encoding='utf-8').splitlines():
    spec = raw_line.split('#', 1)[0].strip()
    if not spec or '==' not in spec: continue
    distribution, expected = (part.strip() for part in spec.split('==', 1))
    if distribution.lower().replace('_', '-') in excluded: continue
    if not version_matches(installed_version(distribution), expected): missing_specs.append(spec)
if missing_specs:
    print(f'Installing {len(missing_specs)} missing/mismatched A1111 packages...')
    run([venv_python, '-m', 'pip', 'install', '--upgrade', *missing_specs])
    environment_changed = True
else:
    print('All ordinary A1111 package versions already match.')

# Binary-sensitive packages require both the correct metadata version and a successful import.
install_if_needed('numpy', '1.26.2', import_code='import numpy; assert numpy.__version__ == "1.26.2"')
install_if_needed('scikit-image', '0.21.0', import_code='import skimage; from skimage import exposure; assert skimage.__version__ == "0.21.0"')
install_if_needed('open-clip-torch', '2.20.0', import_code='import open_clip')
install_if_needed('xformers', '0.0.23.post1', import_code='import torch, xformers; import xformers._C; assert torch.__version__.startswith("2.1.2")')

# OpenAI CLIP has no reliable release version metadata, so its import is the idempotent check.
if probe('import clip').returncode != 0:
    print('Installing the required OpenAI CLIP revision...')
    run([venv_python, '-m', 'pip', 'install', '--no-build-isolation', 'git+https://github.com/openai/CLIP.git@d50d76daa670286dd6cacf3bcd80b5e4823fc8e1'])
    environment_changed = True
else:
    print('Reusing package: OpenAI CLIP')

# CLIP or another legacy installer may alter setuptools; enforce the official A1111 pin only if needed.
install_if_needed('setuptools', '69.5.1', import_code='import pkg_resources')

# A1111 preparation is skipped, so install only ControlNet requirements whose constraints are
# not already satisfied. Pin controlnet_aux to the extension's compatible minimum release.
controlnet_specs = []
controlnet_requirement_lines = (controlnet_extension / 'requirements.txt').read_text(encoding='utf-8').splitlines() + ['opencv-python-headless==4.9.0.80', 'opencv-contrib-python==4.9.0.80']
for raw_line in controlnet_requirement_lines:
    spec = raw_line.split('#', 1)[0].strip()
    if not spec: continue
    if spec == 'mediapipe': spec = 'mediapipe==0.10.11'
    elif spec.startswith('opencv-python>='): spec = 'opencv-python==4.9.0.80'
    elif spec.startswith('timm<='): spec = 'timm==0.6.7'
    elif spec.startswith('controlnet_aux'): spec = 'controlnet_aux==0.0.9'
    check = probe(f'from importlib.metadata import version; from packaging.requirements import Requirement; r=Requirement({spec!r}); assert r.specifier.contains(version(r.name), prereleases=True)')
    if check.returncode != 0: controlnet_specs.append(spec)
if controlnet_specs:
    print(f'Installing {len(controlnet_specs)} missing/mismatched ControlNet packages...')
    controlnet_constraints = Path('/content/a1111_controlnet_constraints.txt')
    controlnet_constraints.write_text('numpy==1.26.2\ntorch==2.1.2\ntorchvision==0.16.2\nopencv-python==4.9.0.80\nopencv-python-headless==4.9.0.80\nopencv-contrib-python==4.9.0.80\n', encoding='utf-8')
    run([venv_python, '-m', 'pip', 'install', '--upgrade', '--constraint', controlnet_constraints, *controlnet_specs])
    environment_changed = True
else:
    print('All ControlNet package requirements already match.')
# Re-check binary-sensitive packages after all dependency resolution.
install_if_needed('numpy', '1.26.2', import_code='import numpy; assert numpy.__version__ == "1.26.2"')
install_if_needed('scikit-image', '0.21.0', import_code='import skimage; from skimage import exposure; assert skimage.__version__ == "0.21.0"')
install_if_needed('torch', '2.1.2', 'torch==2.1.2', 'import torch; assert torch.__version__.startswith("2.1.2")', ['--index-url', 'https://download.pytorch.org/whl/cu121'])
install_if_needed('torchvision', '0.16.2', 'torchvision==0.16.2', 'import torchvision; assert torchvision.__version__.startswith("0.16.2")', ['--index-url', 'https://download.pytorch.org/whl/cu121'])
install_if_needed('xformers', '0.0.23.post1', import_code='import torch, xformers; import xformers._C; assert torch.__version__.startswith("2.1.2")')
install_if_needed('setuptools', '69.5.1', import_code='import pkg_resources')
controlnet_health = probe('import cv2, mediapipe, controlnet_aux; from controlnet_aux import SamDetector; assert hasattr(mediapipe, "solutions")')
if controlnet_health.returncode != 0:
    print('CONTROLNET PACKAGE VALIDATION FAILED:\n' + controlnet_health.stderr)
    raise RuntimeError('ControlNet dependency validation failed after restoring the pinned binary stack.')
runtime_health = 'import pkg_resources, numpy, torch, torchvision, skimage, gradio, clip, open_clip, pytorch_lightning, xformers, controlnet_aux; import xformers._C; from controlnet_aux import SamDetector; from skimage import exposure; from taming.modules.vqvae.quantize import VectorQuantizer2; assert numpy.__version__ == "1.26.2"; assert torch.__version__.startswith("2.1.2"); assert torchvision.__version__.startswith("0.16.2"); assert skimage.__version__ == "0.21.0"'
health = probe(runtime_health)
if health.returncode != 0:
    print('FINAL RUNTIME VALIDATION FAILED:\n' + health.stderr)
    raise RuntimeError('A1111 package validation failed before launch; the exact failing import is printed above.')
print('A1111 runtime validation passed.')

# Download helpers. Existing valid files are never downloaded again.
try:
    from google.colab import userdata
    CIVITAI_TOKEN = userdata.get('CIVITAI_TOKEN')
except Exception:
    CIVITAI_TOKEN = os.environ.get('CIVITAI_TOKEN', '')

def civitai_headers():
    return {'Authorization': f'Bearer {CIVITAI_TOKEN}'} if CIVITAI_TOKEN else {}

def require_token_if_needed():
    global CIVITAI_TOKEN
    if not CIVITAI_TOKEN:
        CIVITAI_TOKEN = getpass('CivitAI token is required for first-time downloads (input hidden): ').strip()
    if not CIVITAI_TOKEN:
        raise RuntimeError('No CivitAI token supplied. Add CIVITAI_TOKEN in Colab Secrets and rerun.')

def validate_safetensors(path):
    if not nonempty(path, 1024 * 1024):
        return False
    with path.open('rb') as f:
        header_length = int.from_bytes(f.read(8), 'little')
        header = f.read(min(header_length, 128))
    return 0 < header_length < 200_000_000 and header.lstrip().startswith(b'{')

def download_civitai(model_id, destination, expected_types):
    if validate_safetensors(destination):
        print(f'Reusing: {destination.name}')
        return
    destination.unlink(missing_ok=True)
    require_token_if_needed()
    import requests
    try:
        info = requests.get(f'https://civitai.com/api/v1/models/{model_id}', headers=civitai_headers(), timeout=60)
        info.raise_for_status()
        info = info.json()
        if str(info.get('type', '')).upper() not in expected_types:
            raise ValueError(f'Unexpected CivitAI type: {info.get("type")}')
        versions = info.get('modelVersions') or []
        files = [f for v in versions for f in v.get('files', []) if f.get('name', '').lower().endswith('.safetensors')]
        file = next((f for f in files if f.get('primary')), files[0] if files else None)
        if not file or not file.get('downloadUrl'):
            raise ValueError('No safetensors download is available for this model.')
        temp = destination.with_suffix(destination.suffix + '.part')
        temp.unlink(missing_ok=True)
        with requests.get(file['downloadUrl'], headers=civitai_headers(), stream=True, allow_redirects=True, timeout=(30, 1800)) as response:
            response.raise_for_status()
            total = int(response.headers.get('content-length', 0))
            downloaded = 0
            with temp.open('wb') as output:
                for chunk in response.iter_content(8 * 1024 * 1024):
                    if chunk:
                        output.write(chunk); downloaded += len(chunk)
                        if total: print(f'  {destination.name}: {downloaded / total:.0%}', end='\r')
        print()
        temp.replace(destination)
        if not validate_safetensors(destination):
            raise ValueError('Downloaded file is not a valid safetensors model.')
        print(f'Downloaded: {destination.name}')
    except Exception as exc:
        destination.unlink(missing_ok=True)
        raise RuntimeError(f'Download failed for CivitAI model {model_id}. Open its page, accept required terms, verify CIVITAI_TOKEN, then rerun. Details: {exc}') from exc

CHECKPOINT = MODELS_DIR / 'epicrealism_naturalSinRC1VAE.safetensors'
download_civitai(25694, CHECKPOINT, {'CHECKPOINT'})
for model_id, filename in [(118893, 'curvy_lora.safetensors'), (143323, 'thick_lora.safetensors'), (134883, 'realistic_skin_lora.safetensors')]:
    download_civitai(model_id, LORAS_DIR / filename, {'LORA', 'LOCON', 'LYCORIS'})

# Install missing ControlNet models only. Hugging Face token is optional unless the host asks for one.
def download_hf(repo, filename, destination):
    if nonempty(destination, 1024 * 1024):
        print(f'Reusing: {destination.name}'); return
    try:
        destination.parent.mkdir(parents=True, exist_ok=True)
        run([venv_python, '-m', 'pip', 'install', 'huggingface_hub'])
        result = subprocess.run([venv_python, '-c', f'from huggingface_hub import hf_hub_download; print(hf_hub_download(repo_id={repo!r}, filename={filename!r}))'], text=True, capture_output=True, check=True)
        cached = Path(result.stdout.strip().splitlines()[-1]); shutil.copy2(cached, destination)
        print(f'Downloaded: {destination.name}')
    except Exception as exc:
        raise RuntimeError(f'Hugging Face download failed for {filename}. Check internet/licence access and rerun. Details: {exc}') from exc

download_hf('lllyasviel/ControlNet-v1-1', 'control_v11p_sd15_openpose.pth', CONTROLNET_DIR / 'control_v11p_sd15_openpose.pth')
download_hf('lllyasviel/ControlNet-v1-1', 'control_v11p_sd15_canny.pth', CONTROLNET_DIR / 'control_v11p_sd15_canny.pth')
download_hf('h94/IP-Adapter', 'models/ip-adapter-plus-face_sd15.bin', CONTROLNET_DIR / 'ip-adapter-plus-face_sd15.bin')
download_hf('h94/IP-Adapter', 'models/image_encoder/pytorch_model.bin', controlnet_extension / 'annotator' / 'downloads' / 'clip_vision' / 'clip_h.pth')

# Safe persistent links: A1111 reads/writes the dedicated Drive folders.
def link_folder(link, target):
    target.mkdir(parents=True, exist_ok=True)
    if link.is_symlink():
        if link.resolve() == target.resolve(): return
        link.unlink()
    elif link.exists():
        contents = list(link.iterdir()) if link.is_dir() else []
        allowed = {'Put Stable Diffusion checkpoints here.txt', 'Put Loras here.txt'}
        if any(item.name not in allowed for item in contents):
            raise RuntimeError(f'Refusing to replace non-empty folder {link}. Move its files to {target} once, then rerun.')
        for item in contents: item.unlink()
        link.rmdir()
    link.parent.mkdir(parents=True, exist_ok=True)
    link.symlink_to(target, target_is_directory=True)

link_folder(WEBUI_DIR / 'models' / 'Stable-diffusion', MODELS_DIR)
link_folder(WEBUI_DIR / 'models' / 'Lora', LORAS_DIR)
link_folder(WEBUI_DIR / 'outputs', OUTPUTS_DIR)

config = {'sd_model_checkpoint': CHECKPOINT.name, 'sampler_name': 'DPM++ 2M Karras', 'steps': 30, 'cfg_scale': 7.5, 'width': 768, 'height': 1024, 'negative_prompt': 'skinny, thin, deformed, extra limbs, bad hands, distorted face, blurry, low quality, cartoon, illustration, 3D render, ugly, unnatural proportions', 'output_directory': str(OUTPUTS_DIR), 'outdir_txt2img_samples': str(OUTPUTS_DIR / 'txt2img-images'), 'outdir_img2img_samples': str(OUTPUTS_DIR / 'img2img-images')}
(WEBUI_DIR / 'config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
controlnet_template = {'identity': {'folder': str(REFERENCE_DIR / 'identity'), 'module': 'ip-adapter_clip_sd15', 'model': 'ip-adapter-plus-face_sd15.bin', 'weight': 0.70}, 'style_or_clothing': {'folder': str(REFERENCE_DIR / 'style'), 'module': 'reference_adain+attn', 'model': 'None', 'weight': 0.45}, 'pose': {'folder': str(REFERENCE_DIR / 'pose'), 'module': 'openpose_full', 'model': 'control_v11p_sd15_openpose.pth', 'weight': 0.65}}
(PROJECT_DIR / 'controlnet.json').write_text(json.dumps(controlnet_template, indent=2), encoding='utf-8')

# Save the fully validated environment before starting the long-running WebUI process.
# This makes the cache durable even when the launch cell is stopped after receiving its URL.
if environment_changed or not VENV_ARCHIVE.exists():
    VENV_ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
    archive_temp = VENV_ARCHIVE.with_suffix('.tar.part')
    archive_temp.unlink(missing_ok=True)
    print('Saving the validated package environment for future runtimes...')
    run(['tar', '-C', '/content', '-cf', archive_temp, RUNTIME_VENV_DIR.name])
    archive_temp.replace(VENV_ARCHIVE)
    print(f'Package cache saved: {VENV_ARCHIVE}')
else:
    print(f'Reusing validated package cache: {VENV_ARCHIVE}')

# Launch in the foreground so all output and the public URL appear in this cell.
os.environ.update({'PYTHONUNBUFFERED': '1', 'MPLBACKEND': 'Agg', 'WEBUI_LAUNCH_LIVE_OUTPUT': '1', 'STABLE_DIFFUSION_REPO': SD_REPO, 'STABLE_DIFFUSION_COMMIT_HASH': SD_COMMIT, 'python_cmd': '/usr/bin/python3.10'})
print('\n' + '=' * 78)
print('STARTING A1111 — wait for: Running on public URL: https://...gradio.live')
print('=' * 78 + '\n')
launch_command = [venv_python, '-u', 'launch.py', '-f', '--skip-prepare-environment', '--share', '--xformers', '--medvram', '--no-half-vae', '--enable-insecure-extension-access', '--api', '--theme', 'dark', '--data-dir', WEBUI_DIR, '--controlnet-dir', CONTROLNET_DIR]
print('A1111 Git/package preparation is disabled; using the explicit saved repositories and local pinned packages.')
process = subprocess.Popen(list(map(str, launch_command)), cwd=str(WEBUI_DIR), env=os.environ.copy(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
assert process.stdout is not None
for line in process.stdout:
    print(line, end='', flush=True)
exit_code = process.wait()
if exit_code:
    raise RuntimeError(f'A1111 stopped with exit code {exit_code}. Read the final lines above for the exact error.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Reusing saved A1111 WebUI from Drive — no Git update or checkout will run.
Reusing saved ControlNet extension from Drive.
Reusing saved dependency: stable-diffusion-webui-assets
Reusing saved dependency: stable-diffusion-stability-ai
Reusing saved dependency: generative-models
Reusing saved dependency: k-diffusion
Reusing saved dependency: BLIP
Reusing saved dependency: taming-transformers
Reusing local /content WebUI packages for this runtime.
Reusing package: torch 2.1.2+cu121
Reusing package: torchvision 0.16.2+cu121
Installing 1 missing/mismatched A1111 packages...

$ /content/image_generation_project_venv/bin/python -m pip install --upgrade httpcore==0.15
Installing required numpy 1.26.2; found 2.2.6.

$ /content/image_generation_project_venv/bin/python -m pip install --upgrade numpy==1.26.2
Reusing package: scikit-image 0.21.0
Reusing package: open-clip

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Moved failed partial rebuild to: /content/drive/MyDrive/Image_Generation_Project/webui_failed_rebuild_20260819_052935
Restored previous WebUI: /content/drive/MyDrive/Image_Generation_Project/webui

Starting restored A1111. Wait for the public Gradio URL.

Launching Web UI with arguments: -f --skip-prepare-environment --share --xformers --medvram --no-half-vae --enable-insecure-extension-access --api --theme dark --data-dir /content/drive/MyDrive/Image_Generation_Project/webui --controlnet-dir /content/drive/MyDrive/Image_Generation_Project/controlnet_models


## Use the WebUI

**New virtual person:** Use `txt2img` with a portrait description. Start with: `portrait photo of a new virtual adult person, natural skin texture, realistic lighting, 85mm lens, high detail`. Optional LoRAs: `<lora:curvy_lora:0.65> <lora:thick_lora:0.45> <lora:realistic_skin_lora:0.35>`.

**Existing virtual person:** Save generated reference portraits in `person_models/<name>/` or `reference_images/identity/`. In `img2img`, select the reference and use denoising strength around `0.30–0.40`. For stronger pose/identity guidance, use the ControlNet panel with IP-Adapter Face or OpenPose.

**Later sessions:** choose GPU, run the same code cell, authorise Drive if asked, and wait for a new Gradio URL. It skips existing WebUI files and model downloads. Keep the cell running while using the URL.